# Assignment 2 — Data Pipeline and Preparation

**Unit:** COMP5310 — Project Stage 2  
**Dataset:** NSW Train Occupancy  
**Purpose:** Reproduce the data preparation process established in Assignment 1 and prepare a reliable dataset for supervised classification.

## 1. Pipeline objectives

This notebook will:

1. Load the original NSW Train Occupancy dataset using a relative file path.
2. Audit the data structure, missing values, duplicates, variable types and inconsistent values.
3. Apply documented cleaning and validation steps.
4. Construct the binary target `Crowded` from `Occupancy Status`.
5. Save the prepared dataset for the modelling and evaluation notebook.
6. Record the assumptions, limitations and potential sources of target leakage.

The pipeline must run from beginning to end without manual editing. The target definition, predictor availability and evaluation strategy must be confirmed before model fitting.

## 2. Reproducibility

The pipeline uses relative paths so that the project can be run from the submitted project structure. Cleaning decisions will be documented, and the resulting data will be checked before it is saved.

## 3. Stakeholder context

The intended stakeholder is a public transport operations team. The analysis aims to identify train observations associated with passenger crowding so that potentially problematic services and locations can be prioritised for further investigation. Predictions are decision-support signals, not proof of the causes of crowding.

In [8]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 4. Load the raw dataset

The original NSW Train Occupancy CSV is loaded from a relative path. The notebook checks the expected project locations so that it can be run from either the project root or the notebook directory.

In [9]:
# Locate the raw dataset using relative paths
possible_paths = [
    Path("data/raw/train_occupancy.csv"),
    Path("../../data/raw/train_occupancy.csv"),
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Could not find train_occupancy.csv. "
        "Check the project folder and relative path."
    )

train_raw = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print(f"File location: {data_path}")
print(f"Rows: {train_raw.shape[0]:,}")
print(f"Columns: {train_raw.shape[1]}")

Dataset loaded successfully.
File location: ../../data/raw/train_occupancy.csv
Rows: 50,250
Columns: 14


## 5. Initial data audit

Before cleaning, we inspect the dataset's dimensions, variable types, missing values, duplicate records and category consistency. This establishes evidence for subsequent cleaning decisions and helps prevent unnecessary or unjustified changes to the data.

In [10]:
# 5.1 Inspect the structure
print("Dataset dimensions:", train_raw.shape)

display(train_raw.head())

print("\nVariable types:")
display(
    train_raw.dtypes.rename("data_type").to_frame()
)

# 5.2 Missing values
missing_audit = pd.DataFrame({
    "missing_count": train_raw.isna().sum(),
    "missing_percent": (
        train_raw.isna().mean() * 100
    ).round(2)
}).sort_values("missing_count", ascending=False)

print("\nMissing-value audit:")
display(missing_audit)

# 5.3 Exact duplicate rows
duplicate_count = train_raw.duplicated().sum()

print(f"\nExact duplicate rows after the first occurrence: {duplicate_count:,}")

# 5.4 Number of distinct values in each variable
print("\nDistinct values per variable:")
display(
    train_raw.nunique(dropna=False)
    .rename("distinct_values")
    .to_frame()
)

Dataset dimensions: (50250, 14)


,day,Actual.Stop.Station,Actual.Station.Arrv.Time,Actual.Station.Dprt.Time,Segment.Direction,Trip.Name,Service.Line,Orig..Station,Dest..Station,Leading.Set.Type,Node.Seq.Order,Actual.Station.Dprt.Time.Band,Occupancy Status,Occupancy Range
0,9,Miranda,2017-01-10 00:18:17,2017-01-10 00:19:04,Down,620S,Illawarra,Central,Cronulla,T,18,00:15-00:29,MANY_SEATS_AVAILABLE,Low: 0-399
1,13,Beecroft,2017-01-13 08:05:59,2017-01-13 08:06:56,Down,152C,North via Macquarie Park,Central,Hornsby,A,16,08:00-08:14,MANY_SEATS_AVAILABLE,Low: 0-399
2,12,Arncliffe,2017-01-12 07:22:51,2017-01-12 07:23:36,Down,607B,Illawarra,Central,Waterfall,T,6,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
3,13,Dulwich Hill,2017-01-13 07:20:39,2017-01-13 07:21:53,Down,58-F,Bankstown,Central,Birrong,A,7,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
4,11,Belmore,2017-01-11 18:38:28,2017-01-11 18:39:07,Up,33-P,Bankstown,Birrong,Central,K,7,18:30-18:44,MANY_SEATS_AVAILABLE,Low: 0-399



Variable types:


,data_type
day,int64
Actual.Stop.Station,str
Actual.Station.Arrv.Time,str
Actual.Station.Dprt.Time,str
Segment.Direction,str
Trip.Name,str
Service.Line,str
Orig..Station,str
Dest..Station,str
Leading.Set.Type,str



Missing-value audit:


,missing_count,missing_percent
Orig..Station,80,0.16
Segment.Direction,78,0.16
Leading.Set.Type,77,0.15
Actual.Stop.Station,74,0.15
Occupancy Status,73,0.15
Dest..Station,63,0.13
Service.Line,59,0.12
Actual.Station.Dprt.Time,11,0.02
Actual.Station.Arrv.Time,7,0.01
day,0,0.00



Exact duplicate rows after the first occurrence: 250

Distinct values per variable:


,distinct_values
day,8
Actual.Stop.Station,308
Actual.Station.Arrv.Time,47199
Actual.Station.Dprt.Time,43140
Segment.Direction,5
Trip.Name,3587
Service.Line,31
Orig..Station,68
Dest..Station,69
Leading.Set.Type,11


## 6. Inspect occupancy categories

The occupancy labels are inspected before cleaning to identify missing values, spelling inconsistencies and unexpected categories. The binary target will be created only after the occupancy labels have been standardised and records with unknown outcomes have been handled.

In [11]:
# Inspect the raw occupancy labels
print("Occupancy Status values:")
display(
    train_raw["Occupancy Status"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nSegment Direction values:")
display(
    train_raw["Segment.Direction"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nWhitespace check for selected categorical fields:")
for col in ["Actual.Stop.Station", "Service.Line", "Segment.Direction"]:
    values = train_raw[col].dropna().astype(str)
    whitespace_count = values.ne(values.str.strip()).sum()
    print(f"{col}: {whitespace_count} values with leading/trailing whitespace")

Occupancy Status values:


,label,count
0,MANY_SEATS_AVAILABLE,48008
1,FEW_SEATS_AVAILABLE,1596
2,STANDING_ROOM_ONLY,398
3,MANY_SEATSAVAILABLE,165
4,NaN,73
5,few_seats_available,8
6,VTANDING_ROOM_ONLY,2



Segment Direction values:


,label,count
0,Up,25197
1,Down,24817
2,Up,80
3,Down,78
4,NaN,78



Whitespace check for selected categorical fields:
Actual.Stop.Station: 0 values with leading/trailing whitespace
Service.Line: 7 values with leading/trailing whitespace
Segment.Direction: 158 values with leading/trailing whitespace


In [12]:
# Create a working copy so the raw dataset remains unchanged
train_df = train_raw.copy()

# 1. Remove leading/trailing whitespace from selected text columns
text_columns = train_df.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    train_df[col] = train_df[col].str.strip()

# 2. Standardise inconsistent occupancy labels
occupancy_mapping = {
    "MANY_SEATSAVAILABLE": "MANY_SEATS_AVAILABLE",
    "few_seats_available": "FEW_SEATS_AVAILABLE",
    "VTANDING_ROOM_ONLY": "STANDING_ROOM_ONLY"
}

train_df["Occupancy Status"] = (
    train_df["Occupancy Status"].replace(occupancy_mapping)
)

# 3. Verify the standardised labels
print("Occupancy categories after standardisation:")
display(
    train_df["Occupancy Status"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nDirection categories after whitespace cleaning:")
display(train_df["Segment.Direction"].value_counts(dropna=False))

print("\nDataset shape before removing missing outcomes or duplicates:")
print(train_df.shape)

Occupancy categories after standardisation:


,label,count
0,MANY_SEATS_AVAILABLE,48173
1,FEW_SEATS_AVAILABLE,1604
2,STANDING_ROOM_ONLY,400
3,NaN,73



Direction categories after whitespace cleaning:


Segment.Direction
Up      25277
Down    24895
NaN        78
Name: count, dtype: int64


Dataset shape before removing missing outcomes or duplicates:
(50250, 14)


In [13]:
# Remove records with missing occupancy status
rows_before = len(train_df)

train_df = train_df.dropna(subset=["Occupancy Status"]).copy()

rows_after = len(train_df)
rows_removed = rows_before - rows_after

print(f"Rows before removal: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows remaining: {rows_after}")

print("\nOccupancy status after removing missing outcomes:")
display(train_df["Occupancy Status"].value_counts(dropna=False))

Rows before removal: 50250
Rows removed: 73
Rows remaining: 50177

Occupancy status after removing missing outcomes:


Occupancy Status
MANY_SEATS_AVAILABLE    48173
FEW_SEATS_AVAILABLE      1604
STANDING_ROOM_ONLY        400
Name: count, dtype: int64

In [14]:
# Identify and remove exact duplicate observations
rows_before = len(train_df)

duplicate_count = train_df.duplicated().sum()
print(f"Exact duplicate rows identified: {duplicate_count}")

train_df = train_df.drop_duplicates().copy()

rows_after = len(train_df)

print(f"Rows before deduplication: {rows_before}")
print(f"Rows removed: {rows_before - rows_after}")
print(f"Rows after deduplication: {rows_after}")
print(f"Dataset dimensions: {train_df.shape}")

Exact duplicate rows identified: 250
Rows before deduplication: 50177
Rows removed: 250
Rows after deduplication: 49927
Dataset dimensions: (49927, 14)
